In [ ]:
pip install sentence-transformers faiss-cpu langchain_community replicate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from langchain_community.llms import Replicate
from google.colab import userdata
import os
import requests
import json

/tmp/ipykernel_1296/719225478.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import Replicate


Assess the LLM’s baseline response to a user quer: at this stage, the LLM generates a baseline response using only its built-in knowledge

In [ ]:
api_token = userdata.get('openrouter_key')
print(f"Retrieved API Token (first 5 chars): {api_token[:5]}...") if api_token else print("API Token is None or empty.")

if not api_token:
  raise ValueError("REPLICATE_api_token not found or is empty in Colab Secrets. Please ensure it's set correctly.")

user_query = "Hi. I’m a citizen of Romania. Do I need a Schengen visa if I travel to France?"
print(f"User Query: {user_query}")

response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": f"Bearer {api_token}",
    },
  data=json.dumps({
    "model": "inclusionai/ling-3.0-flash-vl",
    "messages": [
      {
        "role": "user",
        "content": user_query
      }
    ]
  })
)

print("\nInitial Response (No RAG):\n")
if response.status_code == 200:
  response = response.json()["choices"][0]["message"]["content"]
  print(response)
else:
  print(f"Error: {response.status_code} - {response.text}")


Retrieved API Token (first 5 chars): sk-or...
User Query: Hi. I’m a citizen of Romania. Do I need a Schengen visa if I travel to France?

Initial Response (No RAG):

No, you do not need a Schengen visa to travel to France. Romania is a member of the **Schengen Area** (since 2007), and France is also a Schengen member state. As a Romanian citizen, you can travel freely to France using either your **national ID card** or **passport**, without any visa requirements.

You also enjoy the same freedom of movement rights as any other EU citizen under EU law. Enjoy your trip! 🇷🇴 → 🇫🇷


#RAG setup
In this procedure, I’ll upload GoTravel’s internal documents containing company policies andguidelines. The document is then split into chunks, which are embedded and indexed in Facebook AIsimilarity search (FAISS), and used to provide relevant context that enhances the LLM’s responses.

In [ ]:
from google.colab import files
uploaded = files.upload()
doc_path = list(uploaded.keys())[0]
with open(doc_path, 'r', encoding='utf-8') as f:
  document_text = f.read()

def split_into_chunks(text, chunk_size=300, overlap=50):
  chunks = []
  start = 0
  while start < len(text):
    end = start + chunk_size
    chunks.append(text[start:end])
    start += chunk_size - overlap
  return chunks

chunks = split_into_chunks(document_text)
print(f"Document split into {len(chunks)} chunks.")

Saving External_KB_for_RAG.txt to External_KB_for_RAG.txt
Document split into 10 chunks.


Convert these chunks into vector embeddings and store them in a searchable FAISS index so that the Travel Assistant can retrieve relevant context and provide accurate, context-awareresponses

In [ ]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embed_model.encode(chunks)
dimension = embeddings.shape[1]

# Initialize FAISS index
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# Keep chunks for later retrieval
stored_chunks = chunks
print(f"Stored {len(stored_chunks)} embeddings in FAISS index.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Stored 10 embeddings in FAISS index.


Retrieve relevant chunks from FAISS to generate RAG-enabled response

In [ ]:
query_embedding = embed_model.encode([user_query])
D, I = index.search(query_embedding, k=3)

top_chunks = [stored_chunks[i] for i in I[0]]
context = "\n".join(top_chunks)

print(f"Retrieved {len(top_chunks)} relevant chunks from FAISS index.")

rag_prompt = f"""Use the following context to answer the question.

Context:
{context}

Question:
{user_query}
"""

rag_response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": f"Bearer {api_token}",
    },
  data=json.dumps({
    "model": "inclusionai/ling-3.0-flash-vl",
    "messages": [
      {
        "role": "user",
        "content": user_query
      }
    ]
  })
)

if response.status_code == 200:
  rag_response = (rag_response.json()["choices"][0]["message"]["content"])
else:
  print(f"Error: {response.status_code} - {response.text}")
  raise Exception("Error retrieving RAG response.")

Retrieved 3 relevant chunks from FAISS index.


**Compare baseline responses with context-augmented answers**
In this procedure, I’ll compare the LLM’s baseline response and RAG-enhanced response to assess whether real-time access to GoTravel’s internal documents improved the accuracy and relevance of responses to customer queries.


In [ ]:
from IPython.display import display, Markdown
display(Markdown(f"""
  # Original Query
  **{user_query}**
  --------------
  # Without RAG
  {response}
  ------------------
  # With RAG (Context-Augmented)
  {rag_response}
"""))


  # Original Query
  **Hi. I’m a citizen of Romania. Do I need a Schengen visa if I travel to France?**
  --------------
  # Without RAG
  No, you do not need a Schengen visa to travel to France. Romania is a member of the **Schengen Area** (since 2007), and France is also a Schengen member state. As a Romanian citizen, you can travel freely to France using either your **national ID card** or **passport**, without any visa requirements.

You also enjoy the same freedom of movement rights as any other EU citizen under EU law. Enjoy your trip! 🇷🇴 → 🇫🇷
  ------------------
  # With RAG (Context-Augmented)
  No, you do not need a Schengen visa to travel to France. Romania is a member of both the **European Union** and the **Schengen Area** (since 2007–2008). France is also a Schengen member state.

As a Romanian citizen, you can travel to France — and to any other Schengen country — freely, using either your **Romanian national ID card** or your **passport**. There are no border controls or visa requirements for short stays (up to 90 days within a 180-day period) for purposes such as tourism, business, or transit.

Enjoy your trip to France! 🇷🇴 → 🇫🇷
